# 07 — Narrador Natural (conversa motorista × modelos de demanda)

Responde à pergunta do **motorista**: *"Vale a pena ir ao Aeroporto Salgado Filho (SBPA) no dia X, às Y horas, esperando um pico de corridas?"* — pico originado por **atraso de voos** e **eventos climáticos**. Porta do notebook `03_ANALISE/05_5_Narrador_Natural`, adaptada ao pacote.

**Pré-requisito:** rodar antes o `06_modelos_pareados_demanda.ipynb` (gera os artefatos em `Work/narrador_artefatos/`).

**Fluxo:** interpreta dia+hora (local → UTC) → monta features no mesmo namespace do treino → roda os 2 modelos pareados → decide se há demanda (limiar) → explica via SHAP + contexto histórico de voos/reputação → narra em linguagem natural com a **API da Claude**. Se não há demanda no horário pedido, varre +1h/+2h.

## Onde colocar a chave da Anthropic
A chave **não deve ser versionada**. Use uma das opções (a célula abaixo tenta as duas, nesta ordem):
1. **Variável de ambiente** `ANTHROPIC_API_KEY` (recomendado). No PowerShell: `setx ANTHROPIC_API_KEY "sk-ant-..."` (reabra o terminal) ou, só para a sessão: `$env:ANTHROPIC_API_KEY="sk-ant-..."`.
2. **Arquivo** `cred/anthropic.key` na raiz do pacote (já ignorado pelo `.gitignore` via `*.key`) contendo apenas a chave.

(Opcional) `OPENMETEO_API_KEY` como variável de ambiente, se tiver a chave comercial da Open-Meteo. Sem ela, usa-se a API pública gratuita.

In [ ]:
# (Colab) descomente se precisar instalar as dependências neste ambiente:
# !pip install -q anthropic lightgbm shap joblib requests pandas pyarrow

# Garante que o diretório de trabalho é a raiz do pacote.
import os
from pathlib import Path

while not Path("params.yaml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
print("Diretório de trabalho:", Path.cwd())

In [ ]:
# Carrega a chave da Anthropic: 1) variável de ambiente; 2) arquivo cred/anthropic.key.
import os
from pathlib import Path


def carregar_anthropic_key():
    k = os.environ.get("ANTHROPIC_API_KEY")
    if k:
        return k.strip()
    f = Path("cred/anthropic.key")
    if f.exists():
        return f.read_text(encoding="utf-8").strip()
    return None


ANTHROPIC_API_KEY = carregar_anthropic_key()
print("Chave da Anthropic encontrada:", bool(ANTHROPIC_API_KEY))
if not ANTHROPIC_API_KEY:
    print("  -> sem chave: a ficha técnica funciona, mas a narração em linguagem natural fica desativada.")

In [ ]:
# Instancia o narrador: carrega os 2 modelos pareados + prior + dados de apoio.
from tese_pipeline.narrador.narrador import Narrador

nar = Narrador(
    anthropic_key=ANTHROPIC_API_KEY,
    openmeteo_key=os.environ.get("OPENMETEO_API_KEY"),  # opcional
    # model_llm="claude-haiku-4-5",  # ajuste o modelo se desejar
)

## Consulta por data/hora direta

`consultar(datetime_local, dist_km=...)` — `dist_km` é a distância do motorista ao SBPA (fator de *esforço*: quanto mais longe, mais forte o sinal precisa ser). A ficha técnica (features + valores + SHAP) é impressa sempre; a narração em linguagem natural só com a chave configurada.

In [ ]:
import datetime

_ = nar.consultar(datetime.datetime(2025, 9, 20, 18, 0, 0), dist_km=6)   # perto

In [ ]:
_ = nar.consultar(datetime.datetime(2025, 7, 15, 3, 0, 0), dist_km=22)   # longe: exige sinal mais forte

## Consulta em linguagem natural

Aceita texto livre (a data/hora é interpretada pela Claude — **requer a chave**).

In [ ]:
_ = nar.consultar("Vale a pena ir ao aeroporto amanhã às 18h?", dist_km=8)

In [ ]:
_ = nar.consultar("E depois de amanhã de manhã, às 7h?", dist_km=10)